In [1]:
import random
import json
from dataclasses import dataclass
from typing import List, Dict
import numpy as np
import pandas as pd

# Configuration
N_EXAMPLES = 200
RANDOM_SEED = 42
AZURE_AI_PROJECT_URL = "https://evalsharedfoundry.openai.azure.com/"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [2]:
# Groundedness rubric definition
GROUND_TRUTH_RUBRIC = {
    1: {
        "level": 1,
        "name": "Severely ungrounded / contradictory",
        "description": (
            "The answer is mostly unrelated to the context or directly contradicts it. "
            "Key claims cannot be traced back to the context and/or conflict with it. "
            "Any overlap with the context is incidental rather than substantively supported."
        ),
    },
    2: {
        "level": 2,
        "name": "Largely ungrounded with superficial overlap",
        "description": (
            "The answer mentions entities or topics from the context, but most substantive claims are unsupported. "
            "Contains obvious speculation, invented details, or new facts not justified by the context. "
            "No direct contradictions, but the answer reads more like a guess than a context-based inference."
        ),
    },
    3: {
        "level": 3,
        "name": "Mixed grounded and ungrounded",
        "description": (
            "Some important claims are clearly supported by the context; others are unsupported or vague. "
            "No blatant contradictions, but there is a noticeable mix of grounded and speculative content. "
            "A careful reader would need to double-check several claims against the context."
        ),
    },
    4: {
        "level": 4,
        "name": "Mostly grounded with minor ungrounded elaboration",
        "description": (
            "The main answer to the question is clearly supported by the context. "
            "Additional details go slightly beyond the context but are generic or weakly speculative, without contradicting the context. "
            "A careful reader would judge almost all important claims as verifiable from the context."
        ),
    },
    5: {
        "level": 5,
        "name": "Fully grounded",
        "description": (
            "All key claims in the answer can be directly supported by the context (possibly up to paraphrasing and synonymy). "
            "No speculative, invented, or out-of-context details. "
            "A careful reader could annotate each important claim with a specific span of the context that supports it."
        ),
    },
}

GROUND_TRUTH_RUBRIC_TEXT = """Answer groundedness (1–5 scale)
Groundedness measures how well an answer’s claims can be supported by the provided context, not by external world knowledge. Even if the answer is factually correct, it should be treated as ungrounded if its claims can’t be verified from the context alone.

1 – Severely ungrounded / contradictory
- The answer is mostly unrelated to the context or directly contradicts it.
- Key claims cannot be traced back to the context and/or conflict with it.
- Any overlap with the context is incidental rather than substantively supported.

2 – Largely ungrounded with superficial overlap
- The answer mentions entities or topics from the context, but most substantive claims are unsupported.
- Contains obvious speculation, invented details, or new facts not justified by the context.
- No direct contradictions, but the answer reads more like a guess than a context-based inference.

3 – Mixed grounded and ungrounded
- Some important claims are clearly supported by the context; others are unsupported or vague.
- No blatant contradictions, but there is a noticeable mix of grounded and speculative content.
- A careful reader would need to double-check several claims against the context.

4 – Mostly grounded with minor ungrounded elaboration
- The main answer to the question is clearly supported by the context.
- Additional details go slightly beyond the context but are generic or weakly speculative, without contradicting the context.
- A careful reader would judge almost all important claims as verifiable from the context.

5 – Fully grounded
- All key claims in the answer can be directly supported by the context (possibly up to paraphrasing and synonymy).
- No speculative, invented, or out-of-context details.
- A careful reader could annotate each important claim with a specific span of the context that supports it."""

print(GROUND_TRUTH_RUBRIC)
print("\nRubric narrative:\n" + GROUND_TRUTH_RUBRIC_TEXT)


{1: {'level': 1, 'name': 'Severely ungrounded / contradictory', 'description': 'The answer is mostly unrelated to the context or directly contradicts it. Key claims cannot be traced back to the context and/or conflict with it. Any overlap with the context is incidental rather than substantively supported.'}, 2: {'level': 2, 'name': 'Largely ungrounded with superficial overlap', 'description': 'The answer mentions entities or topics from the context, but most substantive claims are unsupported. Contains obvious speculation, invented details, or new facts not justified by the context. No direct contradictions, but the answer reads more like a guess than a context-based inference.'}, 3: {'level': 3, 'name': 'Mixed grounded and ungrounded', 'description': 'Some important claims are clearly supported by the context; others are unsupported or vague. No blatant contradictions, but there is a noticeable mix of grounded and speculative content. A careful reader would need to double-check severa

In [3]:
# Synthetic knowledge base generation
from dataclasses import dataclass
from typing import Dict


@dataclass
class KBEntry:
    topic_type: str
    name: str
    attributes: Dict[str, str]


KB_ENTRIES: List[KBEntry] = [
    KBEntry("city", "Paris", {"country": "France", "population_m": "2.1", "known_for": "art museums and the Eiffel Tower"}),
    KBEntry("city", "Tokyo", {"country": "Japan", "population_m": "13.9", "known_for": "technology and cuisine"}),
    KBEntry("city", "Nairobi", {"country": "Kenya", "population_m": "4.4", "known_for": "safari gateways"}),
    KBEntry("product", "AuroraPhone X", {"category": "smartphone", "release_year": "2021", "battery_life_hours": "24"}),
    KBEntry("product", "EcoBreeze Air", {"category": "air purifier", "release_year": "2020", "coverage_sqft": "500"}),
    KBEntry("book", "The Silent Orbit", {"author": "Lena Ortiz", "genre": "science fiction", "publication_year": "2018"}),
    KBEntry("book", "Gardens of Glass", {"author": "Amir Patel", "genre": "fantasy", "publication_year": "2022"}),
    KBEntry("research", "GraphLite", {"field": "computer science", "focus": "graph neural networks", "venue": "NeurIPS 2022"}),
    KBEntry("research", "CausalAir", {"field": "epidemiology", "focus": "air quality and respiratory health", "venue": "Lancet 2021"}),
    KBEntry("spacecraft", "Odyssey-3", {"agency": "ESA", "mission": "asteroid survey", "launch_year": "2024"}),
]


def kb_entry_to_context_text(entry: KBEntry) -> str:
    """Render a KB entry as a compact table-like string for retrieval context."""

    lines = ["| field | value |", "| --- | --- |", f"| name | {entry.name} |", f"| type | {entry.topic_type} |"]
    for key, value in entry.attributes.items():
        lines.append(f"| {key} | {value} |")
    return "\n".join(lines)


# Quick check of a rendered context
sample_context = kb_entry_to_context_text(KB_ENTRIES[0])
print(sample_context)


| field | value |
| --- | --- |
| name | Paris |
| type | city |
| country | France |
| population_m | 2.1 |
| known_for | art museums and the Eiffel Tower |


In [4]:
# Question and answer template generation
def generate_question_for_entry(entry: KBEntry) -> str:
    """Create a simple attribute-centric question for the given entry."""

    attribute_keys = list(entry.attributes.keys())
    chosen_attr = random.choice(attribute_keys)
    templates = {
        "country": f"What country is {entry.name} in?",
        "population_m": f"What is the population of {entry.name} in millions?",
        "known_for": f"What is {entry.name} known for?",
        "category": f"What category does the product {entry.name} belong to?",
        "release_year": f"When was {entry.name} released?",
        "battery_life_hours": f"How many hours of battery life does {entry.name} offer?",
        "coverage_sqft": f"What coverage area does {entry.name} support in square feet?",
        "author": f"Who wrote the book {entry.name}?",
        "genre": f"What genre is the book {entry.name}?",
        "publication_year": f"When was the book {entry.name} published?",
        "field": f"What academic field does the work {entry.name} belong to?",
        "focus": f"What does the study {entry.name} focus on?",
        "venue": f"Where was {entry.name} presented or published?",
        "agency": f"Which agency operates {entry.name}?",
        "mission": f"What is the mission of {entry.name}?",
        "launch_year": f"When was {entry.name} launched?",
    }
    return templates.get(chosen_attr, f"What is {chosen_attr} for {entry.name}?")


def generate_answer_for_level(entry: KBEntry, question: str, context_text: str, level: int) -> str:
    """Generate a synthetic answer whose groundedness matches the rubric level."""

    attr_keys = list(entry.attributes.keys())
    target_attr = None
    for key in attr_keys:
        if key in question:
            target_attr = key
            break
    if target_attr is None:
        target_attr = random.choice(attr_keys)
    correct_value = entry.attributes[target_attr]
    name = entry.name

    if level == 5:
        return f"{name} {target_attr.replace('_', ' ')} is {correct_value}."

    if level == 4:
        extras = [
            "It is well regarded in its category.",
            "This detail aligns with its reputation.",
            "It is a notable example in its field.",
        ]
        return f"{name} {target_attr.replace('_', ' ')} is {correct_value}. " + random.choice(extras)

    if level == 3:
        unsupported = random.choice([
            "It recently won several international awards.",
            "Analysts say it may double in scale next year.",
            "Some reports claim it set a record-high metric.",
        ])
        return f"{name} {target_attr.replace('_', ' ')} is {correct_value}, and {unsupported}"

    if level == 2:
        wrong_value = random.choice(["unknown", "not officially disclosed", "expected next year"])
        return f"{name} is often discussed, but its {target_attr.replace('_', ' ')} is {wrong_value}."

    if level == 1:
        return f"Unrelatedly, another subject suggests the opposite of what is asked; {name} focuses on something entirely different."

    return f"{name} {target_attr.replace('_', ' ')} is {correct_value}."


In [5]:
# Dataset construction
def generate_dataset(n_examples: int) -> List[Dict]:
    """Generate synthetic QnA examples with groundedness labels."""

    records: List[Dict] = []
    levels_cycle = [1, 2, 3, 4, 5]
    for idx in range(n_examples):
        entry = random.choice(KB_ENTRIES)
        context_text = kb_entry_to_context_text(entry)
        question = generate_question_for_entry(entry)
        level = levels_cycle[idx % len(levels_cycle)]
        answer = generate_answer_for_level(entry, question, context_text, level)
        record = {
            "id": f"ex_{idx:04d}",
            "context": context_text,
            "question": question,
            "answer": answer,
            "ground_truth_score": level,
            "ground_truth_label_text": f"{GROUND_TRUTH_RUBRIC[level]['name']}: {GROUND_TRUTH_RUBRIC[level]['description']}",
        }
        records.append(record)
    return records


records = generate_dataset(N_EXAMPLES)
df = pd.DataFrame(records)
df.head()


,id,context,question,answer,ground_truth_score,ground_truth_label_text
0,ex_0000,| field | value |\n| --- | --- |\n| name | Tok...,What country is Tokyo in?,"Unrelatedly, another subject suggests the oppo...",1,Severely ungrounded / contradictory: The answe...
1,ex_0001,| field | value |\n| --- | --- |\n| name | Eco...,What category does the product EcoBreeze Air b...,"EcoBreeze Air is often discussed, but its cate...",2,Largely ungrounded with superficial overlap: T...
2,ex_0002,| field | value |\n| --- | --- |\n| name | Nai...,What is Nairobi known for?,"Nairobi country is Kenya, and Some reports cla...",3,Mixed grounded and ungrounded: Some important ...
3,ex_0003,| field | value |\n| --- | --- |\n| name | Cau...,What academic field does the work CausalAir be...,CausalAir field is epidemiology. It is a notab...,4,Mostly grounded with minor ungrounded elaborat...
4,ex_0004,| field | value |\n| --- | --- |\n| name | Gar...,Who wrote the book Gardens of Glass?,Gardens of Glass author is Amir Patel.,5,Fully grounded: All key claims in the answer c...


In [6]:
# Save and inspect
OUTPUT_PATH = "data/synthetic_groundedness_qna.jsonl"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for row in records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved {len(records)} examples to {OUTPUT_PATH}")
print("Ground truth score counts:")
print(df['ground_truth_score'].value_counts().sort_index())

for level in sorted(GROUND_TRUTH_RUBRIC.keys()):
    print(f"\nExamples for level {level}:")
    display(df[df['ground_truth_score'] == level].head(2))


Saved 200 examples to data/synthetic_groundedness_qna.jsonl
Ground truth score counts:
ground_truth_score
1    40
2    40
3    40
4    40
5    40
Name: count, dtype: int64

Examples for level 1:


,id,context,question,answer,ground_truth_score,ground_truth_label_text
0,ex_0000,| field | value |\n| --- | --- |\n| name | Tok...,What country is Tokyo in?,"Unrelatedly, another subject suggests the oppo...",1,Severely ungrounded / contradictory: The answe...
5,ex_0005,| field | value |\n| --- | --- |\n| name | Tok...,What country is Tokyo in?,"Unrelatedly, another subject suggests the oppo...",1,Severely ungrounded / contradictory: The answe...



Examples for level 2:


,id,context,question,answer,ground_truth_score,ground_truth_label_text
1,ex_0001,| field | value |\n| --- | --- |\n| name | Eco...,What category does the product EcoBreeze Air b...,"EcoBreeze Air is often discussed, but its cate...",2,Largely ungrounded with superficial overlap: T...
6,ex_0006,| field | value |\n| --- | --- |\n| name | Aur...,How many hours of battery life does AuroraPhon...,"AuroraPhone X is often discussed, but its batt...",2,Largely ungrounded with superficial overlap: T...



Examples for level 3:


,id,context,question,answer,ground_truth_score,ground_truth_label_text
2,ex_0002,| field | value |\n| --- | --- |\n| name | Nai...,What is Nairobi known for?,"Nairobi country is Kenya, and Some reports cla...",3,Mixed grounded and ungrounded: Some important ...
7,ex_0007,| field | value |\n| --- | --- |\n| name | Cau...,What academic field does the work CausalAir be...,"CausalAir field is epidemiology, and Some repo...",3,Mixed grounded and ungrounded: Some important ...



Examples for level 4:


,id,context,question,answer,ground_truth_score,ground_truth_label_text
3,ex_0003,| field | value |\n| --- | --- |\n| name | Cau...,What academic field does the work CausalAir be...,CausalAir field is epidemiology. It is a notab...,4,Mostly grounded with minor ungrounded elaborat...
8,ex_0008,| field | value |\n| --- | --- |\n| name | Cau...,What does the study CausalAir focus on?,CausalAir focus is air quality and respiratory...,4,Mostly grounded with minor ungrounded elaborat...



Examples for level 5:


,id,context,question,answer,ground_truth_score,ground_truth_label_text
4,ex_0004,| field | value |\n| --- | --- |\n| name | Gar...,Who wrote the book Gardens of Glass?,Gardens of Glass author is Amir Patel.,5,Fully grounded: All key claims in the answer c...
9,ex_0009,| field | value |\n| --- | --- |\n| name | Gra...,Where was GraphLite presented or published?,GraphLite focus is graph neural networks.,5,Fully grounded: All key claims in the answer c...
